# Requirement derivation — EmberSeeker

Derives every requirement threshold that **can** be derived, and states plainly which
ones must stay externally sourced.

**Guiding principle.** A calculation may *derive* a threshold from an externally sourced
higher-level need. It cannot manufacture the need, and it must never be back-filled from
what the design happens to achieve — that is circular and would invalidate the
requirement.

Three tiers are used throughout:

| Tier | Meaning |
|---|---|
| **Calculated** | Falls out of lecture maths from geometry/optics. No external threshold needed. |
| **Derived (external anchor)** | Computed, but consumes an externally cited quantity one level up. |
| **Assumed (external)** | Phenomenology, hardware datasheets or convention. No calculation possible. |

**Decisions applied** (Xav, 1 Nov 2026): R4 dropped; R3 tied to 1 GSD, nadir-referenced;
McArthur Mk5 as the primary fire-spread model with a grassland fast bound and
Vesta Mk2 / AFDRS acknowledged as the modern alternative; elliptical area growth;
LMST to AEDT conversion deprioritised.

**Model scope.** Spherical Earth (R_E = 6378 km), circular Keplerian motion, first-order
secular J2 node precession. No drag, no illumination model, no cloud, no tasking.
Access is geometric only. Inherited from `eo_toolkit` — see its module docstring.

## 0 Setup

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd()))
import eo_toolkit as eo

pd.set_option('display.max_colwidth', 200)
pd.set_option('display.width', 200)

print('eo_toolkit', eo.__version__)

# Collector for the final requirement table.
ROWS = []

def record(rid, quantity, value, tier, basis, lecture=''):
    '''Append one line to the final requirement table.'''
    ROWS.append(dict(ID=rid, Quantity=quantity, Value=value,
                     Tier=tier, Basis=basis, Lecture=lecture))

eo_toolkit N2.4


## 1 External inputs (Tier 3)

Every externally sourced number the derivations consume, in one place. Changing a
requirement means changing a value here, not editing a formula downstream.

Each entry needs its own citation on the slide. The McArthur Mk5 and grassland
coefficients below are the standard published forms (Noble, Bary & Gill 1980, who
tabulated McArthur's meters as equations) — **cite them from the source, not from this
notebook.**

In [2]:
# --- Fire behaviour (Tier 3: external) -----------------------------------------
# McArthur Mk5 forest: head fire rate of spread R [km/h] = 0.0012 * FFDI * w
#   FFDI  McArthur Forest Fire Danger Index [-]
#   w     fine fuel load [t/ha]
FFDI_SEVERE   = 100.0   # 'Catastrophic' band; Black Summer days reached and exceeded this
FFDI_MODERATE = 25.0    # 'High' band; a routine summer afternoon
FUEL_LOAD_THA = 20.0    # dry eucalypt forest fine fuel load [t/ha]

# McArthur Mk4 grassland (fully cured): R [km/h] = 0.13 * GFDI
GFDI_SEVERE = 100.0

LBR = 3.0               # fire ellipse length-to-breadth ratio, wind-driven [-]

A_SUCCESS_HA = 5.0      # initial-attack success definition [ha] (Butler et al. 2026)

T_RESPONSE_H = 2.0      # ground crew response time to preserve [h] (remote VIC/NSW)

# --- Payload (Tier 3: external / comparator class) -----------------------------
PITCH_UM        = 15.0    # detector pixel pitch [um]
APERTURE_M      = 0.10    # entrance aperture diameter [m]
GSD_TARGET_M    = 50.0    # nadir GSD design target [m], FireSat Protoflight class
HALF_FOV_DEG    = 51.34   # payload half field of view [deg]
WAVELENGTH_NM   = 3700.0  # MWIR fire channel centre [nm]
GSD_CROSS_CAP_M = 100.0   # cross-track GSD cap that defines eta_acc [m]

# --- Mission / region (Tier 3: external) ---------------------------------------
LMST_START_H, LMST_END_H = 13.0, 18.5   # diurnal fire window [h LMST] (Giglio 2007; Zhang 2021)
LAT_REGION_DEG  = -37.0   # representative latitude, Vic + southern NSW [deg]
AREA_FRACTION   = 0.90    # fraction of land-cell area R1 must serve [-] (group choice)
SEASON_DAYS     = 90      # 1 Dec 2026 - 28 Feb 2027

# --- Candidate orbits (per CLAUDE.md, 1 Nov 2026) ------------------------------
B0_REVS, B0_DAYS = 15, 1    # B0: single satellite, 1-day repeat
B1_REVS, B1_DAYS = 134, 9   # B1: constellation, 9-day repeat, nearest the comparator

print(f'Severe FFDI {FFDI_SEVERE:.0f}, moderate FFDI {FFDI_MODERATE:.0f}, '
      f'fuel {FUEL_LOAD_THA:.0f} t/ha, LBR {LBR:.1f}')

Severe FFDI 100, moderate FFDI 25, fuel 20 t/ha, LBR 3.0


## 2 Fire growth — the external anchor

McArthur Mk5 gives a *quasi-steady head-fire* rate of spread. The fire perimeter is
treated as an ellipse elongated downwind:

$$L = R\,t, \qquad B = L/\mathrm{LBR}, \qquad A = \tfrac{\pi}{4} L B = \frac{\pi R^2 t^2}{4\,\mathrm{LBR}}$$

Inverting for the time to reach a given area:

$$t(A) = \sqrt{\frac{4 A \,\mathrm{LBR}}{\pi R^2}}$$

**Known limitations, carried forward to the slide.** Mk5 is quasi-steady, so applying it
from $t=0$ overstates growth during the ignition acceleration phase; Mk5 is known to
under-predict under severe wind and has been superseded operationally by Vesta Mk2
inside AFDRS; and the elliptical form assumes a steady wind direction.

In [3]:
def ros_mk5_forest(ffdi, fuel_load_tha):
    '''McArthur Mk5 dry eucalypt forest head fire rate of spread [m/h].'''
    return 0.0012 * ffdi * fuel_load_tha * 1e3


def ros_mk4_grass(gfdi):
    '''McArthur Mk4 fully-cured grassland head fire rate of spread [m/h].'''
    return 0.13 * gfdi * 1e3


def area_from_time(ros_m_per_h, t_h, lbr=LBR):
    '''Elliptical burnt area [m^2] after time t_h at a steady head rate.'''
    L = ros_m_per_h * t_h
    return np.pi * L**2 / (4.0 * lbr)


def time_to_area(ros_m_per_h, area_m2, lbr=LBR):
    '''Time [h] for an elliptical fire to reach a given area.'''
    return np.sqrt(4.0 * area_m2 * lbr / (np.pi * ros_m_per_h**2))


A_SUCCESS_M2 = A_SUCCESS_HA * 1e4

scenarios = {
    'Forest, severe (FFDI 100)':    ros_mk5_forest(FFDI_SEVERE, FUEL_LOAD_THA),
    'Forest, moderate (FFDI 25)':   ros_mk5_forest(FFDI_MODERATE, FUEL_LOAD_THA),
    'Grassland, severe (GFDI 100)': ros_mk4_grass(GFDI_SEVERE),
}

growth = pd.DataFrame([
    dict(Scenario=name,
         ROS_m_per_h=round(r, 1),
         ROS_km_per_h=round(r / 1e3, 2),
         t_to_5ha_min=round(time_to_area(r, A_SUCCESS_M2) * 60.0, 1),
         area_at_1h_ha=round(area_from_time(r, 1.0) / 1e4, 2),
         area_at_24h_ha=round(area_from_time(r, 24.0) / 1e4, 1))
    for name, r in scenarios.items()
])
growth

,Scenario,ROS_m_per_h,ROS_km_per_h,t_to_5ha_min,area_at_1h_ha,area_at_24h_ha
0,"Forest, severe (FFDI 100)",2400.0,2.4,10.9,150.80,86858.8
1,"Forest, moderate (FFDI 25)",600.0,0.6,43.7,9.42,5428.7
2,"Grassland, severe (GFDI 100)",13000.0,13.0,2.0,4424.41,2548460.0


### 2.1 What this tells us — and what it does *not* justify

Read the `t_to_5ha_min` column before going further. The time from ignition to the
5-hectare initial-attack success threshold is on the order of **tens of minutes**, under
both severe and moderate forest conditions, and faster still in grassland.

That is far shorter than any revisit interval a LEO satellite can offer. Two consequences,
both negative for the derivation plan, and both reported rather than hidden:

1. **R1's daily cadence cannot be derived from the 5-hectare criterion.** The criterion
   demands observation within tens of minutes, which no single-satellite sun-synchronous
   orbit (or modest constellation) can deliver. Deriving R1 from this anchor would produce
   a requirement that is unachievable by construction — the same defect that caused R4 to
   be dropped. R1's cadence therefore stays justified by the **diurnal fire window** plus
   the single-satellite architecture, i.e. externally sourced.

2. **R2's 25 m^2 must not be derived from fire growth.** Running the growth model forward
   to the first access opportunity gives the fire size at observation — see
   `area_at_24h_ha`, which is orders of magnitude above 25 m^2. Deriving R2 that way would
   *loosen* the sensitivity requirement enormously and destroy the mission rationale. R2
   is a **sensitivity floor** representing the earliest detectable stage, anchored to the
   comparator payload class, not a consequence of growth.

The growth model is still valuable, just not as a threshold source. Its correct roles are
shown below: it quantifies why early detection matters, it bounds what the mission can
honestly promise, and it is the strongest available argument for the constellation (B1)
over the single satellite (B0).

In [4]:
# The growth model's legitimate use: fire size at first observation, by revisit interval.
revisit_h = np.array([1, 3, 6, 8, 12, 24, 48])
sizes = pd.DataFrame({'Revisit interval [h]': revisit_h})
for name, r in scenarios.items():
    sizes[f'{name} [ha]'] = np.round(area_from_time(r, revisit_h) / 1e4, 1)
print('Fire size at first observation vs revisit interval')
print('(worst case: ignition just after a pass)')
sizes

Fire size at first observation vs revisit interval
(worst case: ignition just after a pass)


,Revisit interval [h],"Forest, severe (FFDI 100) [ha]","Forest, moderate (FFDI 25) [ha]","Grassland, severe (GFDI 100) [ha]"
0,1,150.8,9.4,4424.4
1,3,1357.2,84.8,39819.7
2,6,5428.7,339.3,159278.7
3,8,9651.0,603.2,283162.2
4,12,21714.7,1357.2,637115.0
5,24,86858.8,5428.7,2548460.0
6,48,347435.0,21714.7,10193839.8


In [5]:
# Sensitivity of the 5 ha time to the three assumed fire inputs.
sens = []
for label, kw in [('baseline', {}),
                  ('LBR 1.5 (less elongated)', dict(lbr=1.5)),
                  ('LBR 6.0 (strongly wind-driven)', dict(lbr=6.0)),
                  ('fuel 12 t/ha', dict(fuel=12.0)),
                  ('fuel 30 t/ha', dict(fuel=30.0))]:
    lbr = kw.get('lbr', LBR)
    fuel = kw.get('fuel', FUEL_LOAD_THA)
    sens.append(dict(
        Case=label,
        severe_min=round(time_to_area(ros_mk5_forest(FFDI_SEVERE, fuel), A_SUCCESS_M2, lbr) * 60, 1),
        moderate_min=round(time_to_area(ros_mk5_forest(FFDI_MODERATE, fuel), A_SUCCESS_M2, lbr) * 60, 1)))
sens = pd.DataFrame(sens)
print('Time to 5 ha [min]. The conclusion above is insensitive to all three inputs:')
print('every case stays far below any achievable revisit interval.')
sens

Time to 5 ha [min]. The conclusion above is insensitive to all three inputs:
every case stays far below any achievable revisit interval.


,Case,severe_min,moderate_min
0,baseline,10.9,43.7
1,LBR 1.5 (less elongated),7.7,30.9
2,LBR 6.0 (strongly wind-driven),15.5,61.8
3,fuel 12 t/ha,18.2,72.8
4,fuel 30 t/ha,7.3,29.1


In [6]:
# Response-time formulation, for the record: detect early enough to preserve T_RESPONSE_H.
# This is what would have set R2 had the timescales permitted it.
for name, r in scenarios.items():
    t5 = time_to_area(r, A_SUCCESS_M2)
    slack_h = t5 - T_RESPONSE_H
    if slack_h <= 0:
        print(f'{name:32s} t_5ha = {t5*60:5.1f} min  <  response {T_RESPONSE_H:.1f} h '
              f'-> no detection time remains; formulation does not close')
    else:
        print(f'{name:32s} t_5ha = {t5*60:5.1f} min  -> detect by {slack_h*60:.1f} min, '
              f'A = {area_from_time(r, slack_h):.1f} m^2')

Forest, severe (FFDI 100)        t_5ha =  10.9 min  <  response 2.0 h -> no detection time remains; formulation does not close
Forest, moderate (FFDI 25)       t_5ha =  43.7 min  <  response 2.0 h -> no detection time remains; formulation does not close
Grassland, severe (GFDI 100)     t_5ha =   2.0 min  <  response 2.0 h -> no detection time remains; formulation does not close


## 3 Orbit geometry (Calculated — Week 8)

Repeat ground-track altitudes, sun-synchronous inclination and node behaviour for the two
candidates. These are design parameters rather than requirements, but R1's access
threshold and eta_acc both depend on them.

In [7]:
def orbit_case(revs, days, label):
    '''Repeat-track solution plus node behaviour for one candidate.'''
    sol = eo.repeat_solution(revs, days)
    orb = eo.orbit_from_altitude(sol['h_km'], sol['inc_deg'])
    shift = eo.ground_track_shift(orb)
    return dict(Case=label, revs=revs, days=days,
                h_km=round(sol['h_km'], 2),
                inc_deg=round(sol['inc_deg'], 3),
                T_min=round(orb['T'] / 60.0, 3),
                revs_per_day=round(orb['revs_solar'], 4),
                raan_dot_deg_day=round(orb['raan_dot'], 5),
                shift_deg_per_rev=round(shift['dlon_deg'], 4),
                shift_km_equator=round(shift['d_km'], 1)), orb

b0_row, B0 = orbit_case(B0_REVS, B0_DAYS, 'B0 (single satellite)')
b1_row, B1 = orbit_case(B1_REVS, B1_DAYS, 'B1 (constellation)')
orbits = pd.DataFrame([b0_row, b1_row])
orbits

,Case,revs,days,h_km,inc_deg,T_min,revs_per_day,raan_dot_deg_day,shift_deg_per_rev,shift_km_equator
0,B0 (single satellite),15,1,567.03,97.659,96.000,15.0000,0.98565,24.0000,2671.6
1,B1 (constellation),134,9,601.54,97.794,96.716,14.8889,0.98565,24.1791,2691.6


In [8]:
# Record the orbit design parameters.
for row, name in [(b0_row, 'B0'), (b1_row, 'B1')]:
    record(name, 'Repeat ground-track altitude',
           f"{row['h_km']:.2f} km ({row['revs']}/{row['days']} repeat)",
           'Calculated',
           'Coprime repeat ratio with Keplerian period and spherical Earth',
           'Wk8')
    record(name, 'Sun-synchronous inclination', f"{row['inc_deg']:.3f} deg",
           'Calculated', 'First-order secular J2 node rate matched to the mean solar year',
           'Wk8')

## 4 eta_acc — the accepted view angle (Calculated — Week 10)

Slide 4 defined this circularly: R1 said eta_acc was "set by R2", and R2 said it was
"the largest angle meeting 100 m, capped at the payload half-FOV". Reconstructing it
shows the 100 m is a **cross-track GSD cap**, and that it is the binding constraint — the
payload half-FOV never comes into play. So the concept was sound; only the wording was
circular.

**One definition, stated here.** The accepted view angle is the largest angle at which the
cross-track ground sample stays within the GSD cap, subject to the payload half-FOV and
the geometric horizon:

$$\eta_{acc} = \min\left(\eta\ \text{such that}\ \mathrm{GSD}_{cross}(\eta) = \mathrm{GSD}_{cap},\ \ \mathrm{half\ FOV},\ \ \rho\right)$$

where $\rho$ is Earth's angular radius at altitude. The cross-track sample grows as
$\mathrm{GSD}_{cross} = \mathrm{GSD}_{nadir}\,(R_s/h)/\sin\varepsilon$, combining range
growth with surface tilt.

The 100 m cap itself is an **assumed** choice (Tier 3) — it is a factor-of-two degradation
from the 50 m nadir target, and needs its own justification on the slide. Everything
downstream of it is calculated. Any radiometric restriction on the usable angle is a
*performance* result from the payload analysis and is cross-checked against this value; it
does not define it.

In [9]:
def eta_for_gsd_cap(h_km, gsd_nadir_m, cap_m):
    '''Largest view angle [deg] whose cross-track GSD stays within cap_m.'''
    lo, hi = 0.0, eo.earth_angular_radius(h_km) - 1e-6
    if eo.off_nadir_gsd(h_km, gsd_nadir_m, hi)['gsd_cross_m'] <= cap_m:
        return hi                                  # cap never reached: horizon-limited
    for _ in range(80):
        mid = (lo + hi) / 2
        if eo.off_nadir_gsd(h_km, gsd_nadir_m, mid)['gsd_cross_m'] < cap_m:
            lo = mid
        else:
            hi = mid
    return lo


def eta_acc_for(orb, half_fov_deg=HALF_FOV_DEG, cap_m=GSD_CROSS_CAP_M):
    '''Accepted view angle [deg] and the access geometry at that angle.'''
    h = orb['h']
    rho = eo.earth_angular_radius(h)
    gsd_nadir = eo.gsd_from_optics(
        h, PITCH_UM, eo.focal_length_for(h, PITCH_UM, GSD_TARGET_M))['gsd_m']
    eta_gsd = eta_for_gsd_cap(h, gsd_nadir, cap_m)
    eta = min(eta_gsd, half_fov_deg, rho - 1e-6)
    limiter = ('GSD cap' if eta == eta_gsd else
               'half-FOV' if eta == half_fov_deg else 'horizon')
    geom = eo.access_geometry(h, eta_deg=eta)
    stretch = eo.off_nadir_gsd(h, gsd_nadir, eta)
    return dict(h_km=round(h, 2),
                rho_deg=round(rho, 3),
                eta_gsd_cap_deg=round(eta_gsd, 3),
                half_fov_deg=half_fov_deg,
                eta_acc_deg=round(eta, 3),
                limited_by=limiter,
                elevation_at_eta_deg=round(geom['eps'], 2),
                slant_range_km=round(geom['R_s'], 1),
                ground_reach_km=round(geom['ground_reach'], 1),
                gsd_along_m=round(stretch['gsd_along_m'], 1),
                gsd_cross_m=round(stretch['gsd_cross_m'], 1),
                swath_km=round(eo.access_width(h, 0.0, fov_deg=2 * eta), 1))

eta_tbl = pd.DataFrame([dict(Case='B0', **eta_acc_for(B0)),
                        dict(Case='B1', **eta_acc_for(B1))])
print('Cross-check against the deck: slide 14 quotes 69 x 100 m at eta_acc,')
print('slide 6 quotes a 1097 km usable swath for the 601.5 km orbit.')
eta_tbl

Cross-check against the deck: slide 14 quotes 69 x 100 m at eta_acc,
slide 6 quotes a 1097 km usable swath for the 601.5 km orbit.


,Case,h_km,rho_deg,eta_gsd_cap_deg,half_fov_deg,eta_acc_deg,limited_by,elevation_at_eta_deg,slant_range_km,ground_reach_km,gsd_along_m,gsd_cross_m,swath_km
0,B0,567.03,66.687,41.491,51.34,41.491,GSD cap,43.83,785.4,520.9,69.3,100.0,1041.8
1,B1,601.54,66.038,41.295,51.34,41.295,GSD cap,43.77,832.2,549.9,69.2,100.0,1099.7


In [10]:
e0, e1 = eta_acc_for(B0), eta_acc_for(B1)
record('R1', 'Accepted view angle eta_acc',
       f"{e0['eta_acc_deg']:.2f} deg (B0), {e1['eta_acc_deg']:.2f} deg (B1)",
       'Calculated',
       f"Largest angle holding cross-track GSD within the {GSD_CROSS_CAP_M:.0f} m cap; "
       f"limited by {e0['limited_by']}. Half-FOV {HALF_FOV_DEG:.2f} deg and horizon "
       f"({e0['rho_deg']:.1f} deg B0, {e1['rho_deg']:.1f} deg B1) are not binding. "
       'No roll agility assumed.',
       'Wk10')
record('R2', 'Cross-track GSD cap', f'{GSD_CROSS_CAP_M:.0f} m',
       'Assumed (external)',
       'Group choice defining eta_acc: a factor-of-two degradation from the 50 m nadir '
       'target. Needs its own justification on the slide; everything downstream of it '
       'is calculated.', '-')
record('R1', 'Usable swath at eta_acc',
       f"{e0['swath_km']:.0f} km (B0), {e1['swath_km']:.0f} km (B1)",
       'Calculated', 'Spherical access width across the full FOV', 'Wk10')

### 4.1 Strip tiling at the region latitude

How the swath compares with the spacing between successive ground-track strips at the
region latitude. This is the geometric reason access is geometry-limited rather than
window-limited.

**Read this carefully — it is not a coverage verdict.** The test below asks whether a
day's (or a cycle's) strips tile the *entire parallel*. That is the right question for the
9-day repeat, where strips fill in progressively. It is the *wrong* question for the 1-day
repeat, where the same strips recur every day and the region is covered if it happens to
lie within one — the region spans roughly ten degrees of longitude, not the full parallel.
Daily regional coverage for the 1-day repeat therefore depends on where the frozen strips
sit relative to the region, which is what the coverage simulation in
`orbit_planning.ipynb` resolves (and why slide 6 reports 100 % for that orbit). Nothing
here contradicts that; this cell simply does not model it.

In [11]:
checks = []
circ = eo.parallel_circumference(LAT_REGION_DEG)
for name, orb, days in [('B0', B0, B0_DAYS), ('B1', B1, B1_DAYS)]:
    eta = eta_acc_for(orb)['eta_acc_deg']
    swath = eo.access_width(orb['h'], 0.0, fov_deg=2 * eta)
    spacing_cycle = circ / (orb['revs_solar'] * days)   # spacing over the full repeat cycle
    spacing_1day = circ / (orb['revs_solar'] * 1.0)     # spacing if it had to close in 1 day
    checks.append(dict(Case=name,
                       repeat_days=days,
                       eta_acc_deg=round(eta, 2),
                       parallel_circumference_km=round(circ, 1),
                       usable_swath_km=round(swath, 1),
                       spacing_over_cycle_km=round(spacing_cycle, 1),
                       spacing_for_1_day_km=round(spacing_1day, 1),
                       tiles_parallel_in_1_day=bool(swath >= spacing_1day),
                       tiles_parallel_over_cycle=bool(swath >= spacing_cycle)))
print(f'Strip spacing evaluated at latitude {LAT_REGION_DEG:.0f} deg')
print('Columns refer to tiling the FULL parallel - see the note above, not a '
      'regional coverage verdict.')
pd.DataFrame(checks)

Strip spacing evaluated at latitude -37 deg
Columns refer to tiling the FULL parallel - see the note above, not a regional coverage verdict.


,Case,repeat_days,eta_acc_deg,parallel_circumference_km,usable_swath_km,spacing_over_cycle_km,spacing_for_1_day_km,tiles_parallel_in_1_day,tiles_parallel_over_cycle
0,B0,1,41.49,32004.6,1041.8,2133.6,2133.6,False,False
1,B1,9,41.30,32004.6,1099.7,238.8,2149.6,False,True


## 5 Optics (Calculated — Week 10)

The 50 m nadir GSD target is comparator-sourced (Tier 3). Everything downstream of it is
calculated: focal length from the pinhole relation, off-nadir pixel stretch at eta_acc,
and the diffraction check that confirms the aperture is not the limiting element.

In [12]:
optics = []
for name, orb in [('B0', B0), ('B1', B1)]:
    h = orb['h']
    f_m = eo.focal_length_for(h, PITCH_UM, GSD_TARGET_M)
    gsd = eo.gsd_from_optics(h, PITCH_UM, f_m)['gsd_m']
    eta = eta_acc_for(orb)['eta_acc_deg']
    stretch = eo.off_nadir_gsd(h, gsd, eta)
    diff = eo.diffraction_limit(h, APERTURE_M, WAVELENGTH_NM)
    regime = eo.resolution_regime(gsd, diff['x_diff_m'])
    optics.append(dict(Case=name,
                       h_km=round(h, 2),
                       eta_acc_deg=round(eta, 2),
                       focal_length_cm=round(f_m * 100, 2),
                       gsd_nadir_m=round(gsd, 2),
                       gsd_along_at_eta_m=round(stretch['gsd_along_m'], 1),
                       gsd_cross_at_eta_m=round(stretch['gsd_cross_m'], 1),
                       pixel_area_ratio=round(stretch['area_ratio'], 2),
                       rayleigh_m=round(diff['x_diff_m'], 2),
                       regime=regime['regime'],
                       rayleigh_over_gsd=round(regime['ratio'], 3)))
optics_tbl = pd.DataFrame(optics)
optics_tbl

,Case,h_km,eta_acc_deg,focal_length_cm,gsd_nadir_m,gsd_along_at_eta_m,gsd_cross_at_eta_m,pixel_area_ratio,rayleigh_m,regime,rayleigh_over_gsd
0,B0,567.03,41.49,17.01,50.0,69.3,100.0,2.77,25.60,sampling pitch larger,0.512
1,B1,601.54,41.30,18.05,50.0,69.2,100.0,2.77,27.15,sampling pitch larger,0.543


In [13]:
o0, o1 = optics[0], optics[1]
record('R2', 'Nadir GSD design target', f'{GSD_TARGET_M:.0f} m',
       'Assumed (external)',
       'FireSat Protoflight payload class, 50 m from ~600 km. Comparator-sourced.', '-')
record('R2', 'Focal length',
       f"{o0['focal_length_cm']:.1f} cm (B0), {o1['focal_length_cm']:.1f} cm (B1)",
       'Calculated', 'Pinhole f = h*p/GSD at the 50 m target and 15 um pitch', 'Wk10')
record('R2', 'GSD at eta_acc (along x cross)',
       f"{o0['gsd_along_at_eta_m']:.0f} x {o0['gsd_cross_at_eta_m']:.0f} m (B0), "
       f"{o1['gsd_along_at_eta_m']:.0f} x {o1['gsd_cross_at_eta_m']:.0f} m (B1)",
       'Calculated', 'Differential angular pixel stretch for cross-track view angle', 'Wk10')
record('R2', 'Diffraction adequacy',
       f"Rayleigh {o0['rayleigh_m']:.1f} m vs GSD {GSD_TARGET_M:.0f} m "
       f"(ratio {o0['rayleigh_over_gsd']:.2f})",
       'Calculated',
       f'Rayleigh 1.22*lambda*h/D at {WAVELENGTH_NM:.0f} nm and D = {APERTURE_M*100:.0f} cm; '
       'sampling pitch is the coarser scale, so the aperture is not limiting',
       'Wk10')

## 6 R3 geolocation — tied to 1 GSD (Derived)

Per Xav's decision, R3's threshold is set by the payload rather than asserted: the
horizontal radial position error must not exceed the resolution element that produced the
detection.

$$\mathrm{RMSE}_{req} = 1 \times \mathrm{GSD}_{nadir}$$

**Nadir-referenced**, deliberately. The local GSD grows off-nadir, so referencing the
nadir value keeps R3 a single number and is the stricter reading at the swath edge. The
published operational figures (OroraTech ~600 m, MyFireWatch 2–5 km) are retained as an
external *sufficiency* check — we sit well inside what ground coordination needs — and no
longer as the source of the number.

In [14]:
gsd_nadir = eo.gsd_from_optics(
    B0['h'], PITCH_UM, eo.focal_length_for(B0['h'], PITCH_UM, GSD_TARGET_M))['gsd_m']
r3_threshold_m = 1.0 * gsd_nadir

local_gsd = []
eta_acc_b0 = eta_acc_for(B0)['eta_acc_deg']
for eta in [0, 10, 20, 30, 40, eta_acc_b0]:
    s = eo.off_nadir_gsd(B0['h'], gsd_nadir, eta)
    local_gsd.append(dict(eta_deg=round(eta, 2),
                          gsd_along_m=round(s['gsd_along_m'], 1),
                          gsd_cross_m=round(s['gsd_cross_m'], 1),
                          local_1gsd_threshold_m=round(s['gsd_cross_m'], 1),
                          adopted_nadir_threshold_m=round(r3_threshold_m, 1)))

print(f'R3 threshold (nadir-referenced, 1 GSD): {r3_threshold_m:.1f} m')
print('Local-GSD alternative shown for comparison only - not adopted.')
pd.DataFrame(local_gsd)

R3 threshold (nadir-referenced, 1 GSD): 50.0 m
Local-GSD alternative shown for comparison only - not adopted.


,eta_deg,gsd_along_m,gsd_cross_m,local_1gsd_threshold_m,adopted_nadir_threshold_m
0,0.00,50.0,50.0,50.0,50.0
1,10.00,50.8,51.8,51.8,50.0
2,20.00,53.5,57.7,57.7,50.0
3,30.00,58.6,69.9,69.9,50.0
4,40.00,67.5,94.4,94.4,50.0
5,41.49,69.3,100.0,100.0,50.0


In [15]:
record('R3', 'Horizontal radial RMSE threshold', f'{r3_threshold_m:.0f} m',
       'Derived (external anchor)',
       'One nadir GSD, so position error does not exceed the resolution element that '
       'produced the detection. Anchor is the comparator-sourced 50 m GSD target. '
       'OroraTech ~600 m / MyFireWatch 2-5 km retained as a sufficiency check only.',
       'Wk10/Wk11')
record('R3', 'Angle reference for the 1-GSD tie', 'Nadir-referenced',
       'Calculated',
       'Local GSD grows off-nadir; nadir referencing keeps one number and is the '
       'stricter reading at the swath edge',
       'Wk10')

## 7 Requirements that stay assumed (Tier 3)

Recorded explicitly so the final table is complete and the slide 4 basis column can be
written straight from it. Each still needs its own citation.

In [16]:
record('R1', 'Revisit cadence', 'At least one eligible access per day',
       'Assumed (external)',
       'NOT derivable from fire growth: see section 2.1. The 5 ha criterion demands '
       'observation within tens of minutes, which no achievable LEO cadence delivers, so '
       'deriving R1 from it would make the requirement unachievable by construction. '
       'Justified instead by the diurnal fire window and the single-satellite architecture.',
       '-')
record('R1', 'Observation window', '13:00-18:30 LMST',
       'Assumed (external)',
       'Diurnal fire cycle peaks after 13:00 and later further south (Giglio 2007; '
       'Zhang et al. 2021). Observational, not derivable.', '-')
record('R1', 'Land-cell area fraction', f'{AREA_FRACTION*100:.0f} %',
       'Assumed (group choice)',
       'A risk-acceptance choice, not a physical quantity. Report sensitivity at '
       '80 / 95 / 99 % to show it is not load-bearing.', '-')
record('R1', 'Season', f'{SEASON_DAYS} days (1 Dec 2026 - 28 Feb 2027)',
       'Assumed (external)', 'BOM Australian fire weather seasons', '-')
record('R2', 'Minimum detectable fire area', '<= 25 m^2 (5 x 5 m)',
       'Assumed (external)',
       'A sensitivity floor representing the earliest detectable stage, anchored to the '
       'FireSat Protoflight published claim. NOT derived from fire growth: running growth '
       'forward to the first access gives hectares (section 2.1), which would loosen the '
       'requirement and destroy the mission rationale. Note the inequality direction: '
       'minimum detectable area <= 25 m^2.', '-')
record('R2', 'Fire and background temperatures', '600 K fire, 330 K background',
       'Assumed (external)', 'Phenomenology (Realmuto et al., HyspIRI saturation study)', '-')
record('R2', 'Detection rule', 'SNR >= 5',
       'Assumed (convention)', 'Community convention, not a derivable quantity', '-')
record('R2', 'Noise terms', 'NETD 1 K, clutter 2 K, calibration 1 K, reflected sun 0.7 K',
       'Assumed (external)', 'Hardware and environment; combined in root-sum-square', '-')
record('R3', 'Error budget component accuracies',
       'Star tracker 10"/2", mount 7"/2", GNSS 3.5 m, timing 1 ms, DEM 4 m vertical',
       'Assumed (external)', 'Datasheet and heritage values; see slide 9 references', '-')
record('-', 'Fire spread inputs',
       f'FFDI {FFDI_SEVERE:.0f} severe / {FFDI_MODERATE:.0f} moderate, '
       f'fuel {FUEL_LOAD_THA:.0f} t/ha, LBR {LBR:.1f}',
       'Assumed (external)',
       'McArthur Mk5 coefficients per Noble, Bary & Gill (1980). Mk5 under-predicts '
       'under severe wind and is superseded by Vesta Mk2 within AFDRS. Supports context '
       'and the B0/B1 trade, not a threshold.', '-')
record('-', 'Initial-attack success area', f'{A_SUCCESS_HA:.0f} ha',
       'Assumed (external)', 'Butler et al. (2026), Victoria', '-')
record('-', 'Ground response time', f'{T_RESPONSE_H:.0f} h',
       'Assumed (external)',
       'Remote VIC/NSW crew response. Formulation did not close against the fire '
       'timescale - see section 2.', '-')
print(f'{len(ROWS)} rows recorded')

25 rows recorded


## 8 Final requirement table

Calculated and derived rows first, then the assumed rows. This is the content for the
slide 4 basis column and the STM.

In [17]:
reqs = pd.DataFrame(ROWS)
order = {'Calculated': 0, 'Derived (external anchor)': 1}
reqs['_k'] = reqs['Tier'].map(lambda t: order.get(t, 2))
reqs = reqs.sort_values(['_k', 'ID', 'Quantity']).drop(columns='_k').reset_index(drop=True)
reqs

,ID,Quantity,Value,Tier,Basis,Lecture
0,B0,Repeat ground-track altitude,567.03 km (15/1 repeat),Calculated,Coprime repeat ratio with Keplerian period and spherical Earth,Wk8
1,B0,Sun-synchronous inclination,97.659 deg,Calculated,First-order secular J2 node rate matched to the mean solar year,Wk8
2,B1,Repeat ground-track altitude,601.54 km (134/9 repeat),Calculated,Coprime repeat ratio with Keplerian period and spherical Earth,Wk8
3,B1,Sun-synchronous inclination,97.794 deg,Calculated,First-order secular J2 node rate matched to the mean solar year,Wk8
4,R1,Accepted view angle eta_acc,"41.49 deg (B0), 41.30 deg (B1)",Calculated,"Largest angle holding cross-track GSD within the 100 m cap; limited by GSD cap. Half-FOV 51.34 deg and horizon (66.7 deg B0, 66.0 deg B1) are not binding. No roll agility assumed.",Wk10
5,R1,Usable swath at eta_acc,"1042 km (B0), 1100 km (B1)",Calculated,Spherical access width across the full FOV,Wk10
6,R2,Diffraction adequacy,Rayleigh 25.6 m vs GSD 50 m (ratio 0.51),Calculated,"Rayleigh 1.22*lambda*h/D at 3700 nm and D = 10 cm; sampling pitch is the coarser scale, so the aperture is not limiting",Wk10
7,R2,Focal length,"17.0 cm (B0), 18.1 cm (B1)",Calculated,Pinhole f = h*p/GSD at the 50 m target and 15 um pitch,Wk10
8,R2,GSD at eta_acc (along x cross),"69 x 100 m (B0), 69 x 100 m (B1)",Calculated,Differential angular pixel stretch for cross-track view angle,Wk10
9,R3,Angle reference for the 1-GSD tie,Nadir-referenced,Calculated,Local GSD grows off-nadir; nadir referencing keeps one number and is the stricter reading at the swath edge,Wk10


In [18]:
print(reqs.groupby('Tier', as_index=False).size()
          .rename(columns={'size': 'Count'}).to_string(index=False))
print()
print('Calculated or derived:')
for _, r in reqs.iterrows():
    if r.Tier.startswith('Assumed'):
        continue
    print(f'  {r.ID:4s} {r.Quantity:38s} {r.Value}')

                     Tier  Count
     Assumed (convention)      1
       Assumed (external)     12
   Assumed (group choice)      1
               Calculated     10
Derived (external anchor)      1

Calculated or derived:
  B0   Repeat ground-track altitude           567.03 km (15/1 repeat)
  B0   Sun-synchronous inclination            97.659 deg
  B1   Repeat ground-track altitude           601.54 km (134/9 repeat)
  B1   Sun-synchronous inclination            97.794 deg
  R1   Accepted view angle eta_acc            41.49 deg (B0), 41.30 deg (B1)
  R1   Usable swath at eta_acc                1042 km (B0), 1100 km (B1)
  R2   Diffraction adequacy                   Rayleigh 25.6 m vs GSD 50 m (ratio 0.51)
  R2   Focal length                           17.0 cm (B0), 18.1 cm (B1)
  R2   GSD at eta_acc (along x cross)         69 x 100 m (B0), 69 x 100 m (B1)
  R3   Angle reference for the 1-GSD tie      Nadir-referenced
  R3   Horizontal radial RMSE threshold       50 m


## 9 Outcome

**Calculated outright (Week 8 / Week 10):** repeat-track altitudes, sun-synchronous
inclinations, node rates and track shift; eta_acc and the usable swath; focal length; GSD
at nadir and at eta_acc; diffraction adequacy.

eta_acc is now defined once, as the angle holding cross-track GSD within the 100 m cap.
That reproduces the deck's own numbers (69 x 100 m at eta_acc on slide 14, 1097 km usable
swath on slide 6), which confirms the reconstruction: the 100 m in R2's conditions was a
cross-track GSD cap all along, and it binds long before the payload half-FOV. The slide 4
wording was circular, the underlying concept was not. The 100 m cap itself remains an
assumed group choice needing justification.

**Derived from an external anchor:** R3's geolocation threshold, now one nadir GSD rather
than an asserted 50 m.

**Could not be derived, and why.** The fire-spread derivation was intended to put R1's
cadence and R2's area on a calculated footing. It does not, and section 2.1 shows why: the
ignition-to-5-hectare time is tens of minutes in every scenario and sensitivity case
tested, against a best achievable revisit of hours. Deriving R1 from it yields a
requirement unachievable by construction; deriving R2 from it loosens the sensitivity
floor by orders of magnitude. Both stay externally sourced.

That is a usable result rather than a dead end. The growth model now does three jobs on the
slides: it quantifies the early-detection rationale, it bounds what the mission can
honestly promise (fire size at first observation), and it is the strongest argument for the
B1 constellation over the single satellite B0, since shrinking the revisit interval is the
only lever that materially reduces fire size at detection.

**Still open for the slides:** whether to report R1's 90 % as a sensitivity sweep or to
re-weight land cells by historical fire occurrence.